***
# Process BTS on-time flight records

This notebook prepares raw Bureau of Transportation Statistics (BTS) flight data for the capstone project. It removes empty or unneeded columns, removes canceled and diverted flights, checks the data types, and saves a smaller file for later cleaning and modeling.

> **Note:** `YEAR` and `AIRPORT` choose the input and output files. Use the `Origin` and `Dest` columns to identify the actual route for each flight.
***

In [149]:
YEAR = 2024

AIRPORT = "JFK"
# AIRPORT = "ORD"
# AIRPORT = "ATL"

RAW_DATA_FILE = f"../data/bts/raw/{YEAR}/{AIRPORT}.csv"
PROCESSED_DATA_FILE = f"../data/bts/processed/{AIRPORT}_{YEAR}.csv"

***
## Load the raw BTS extract

`low_memory=False` tells pandas to inspect the whole CSV when choosing a data type for each column. Dates, times, categories, and numbers will still need to be converted to the correct types during the cleaning step.
***

In [150]:
import pandas as pd

df = pd.read_csv(RAW_DATA_FILE, low_memory=False)

***
## Remove columns with no observations

A column with no values for any flight cannot help the analysis. Print these column names before removing them so it is easy to see whether a different year or airport has different data available.
***

In [151]:
null_only_columns = df.columns[df.isna().all()].tolist()

print(f"Columns containing only null/NaN values: {len(null_only_columns)}")
print(null_only_columns)

df = df.drop(columns=null_only_columns)
df.shape

Columns containing only null/NaN values: 25
['Div3Airport', 'Div3AirportID', 'Div3AirportSeqID', 'Div3WheelsOn', 'Div3TotalGTime', 'Div3LongestGTime', 'Div3WheelsOff', 'Div3TailNum', 'Div4Airport', 'Div4AirportID', 'Div4AirportSeqID', 'Div4WheelsOn', 'Div4TotalGTime', 'Div4LongestGTime', 'Div4WheelsOff', 'Div4TailNum', 'Div5Airport', 'Div5AirportID', 'Div5AirportSeqID', 'Div5WheelsOn', 'Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff', 'Div5TailNum', 'Unnamed: 109']


(246682, 85)

***
## Remove redundant identifiers and out-of-scope fields

Remove mostly empty code-share fields, extra numeric IDs, repeated location details, diversion details, and administrative fields. Keep the readable airline and airport codes for use in later analysis.

`errors="ignore"` lets this code work when some files do not contain every listed column. It also hides misspelled column names, so review this list when the source data or project needs change.
***

In [152]:
columns_to_drop = [
    # Originally scheduled code share (almost entirely null)
    "Originally_Scheduled_Code_Share_Airline",
    "DOT_ID_Originally_Scheduled_Code_Share_Airline",
    "IATA_Code_Originally_Scheduled_Code_Share_Airline",
    "Flight_Num_Originally_Scheduled_Code_Share_Airline",

    "DOT_ID_Reporting_Airline",
    "IATA_CODE_Reporting_Airline",

    # Airline IDs (keep Operating_Airline instead)
    "DOT_ID_Marketing_Airline",
    "IATA_Code_Marketing_Airline",
    "DOT_ID_Operating_Airline",
    "IATA_Code_Operating_Airline",

    # Airport identifiers (keep Origin and Dest)
    "OriginAirportID",
    "OriginAirportSeqID",
    "OriginCityMarketID",
    "OriginStateFips",
    "OriginWac",

    "DestAirportID",
    "DestAirportSeqID",
    "DestCityMarketID",
    "DestStateFips",
    "DestWac",

    # Geographic fields (unless doing geographic analysis)
    "OriginCityName",
    "OriginStateName",

    "DestCityName",
    "DestStateName",

    # Diversion-specific fields
    "DivAirportLandings",
    "DivReachedDest",
    "DivActualElapsedTime",
    "DivArrDelay",
    "DivDistance",

    "Div1Airport",
    "Div1AirportID",
    "Div1AirportSeqID",
    "Div1WheelsOn",
    "Div1TotalGTime",
    "Div1LongestGTime",
    "Div1WheelsOff",
    "Div1TailNum",

    "Div2Airport",
    "Div2AirportID",
    "Div2AirportSeqID",
    "Div2WheelsOn",
    "Div2TotalGTime",
    "Div2LongestGTime",
    "Div2WheelsOff",
    "Div2TailNum",

    # Administrative
    "Flights",
    "Duplicate",

    "Div3Airport", "Div3AirportID", "Div3AirportSeqID", "Div3WheelsOn", "Div3TotalGTime", "Div3LongestGTime", "Div3WheelsOff", "Div3TailNum", "Div4Airport", "Div4AirportID", "Div4AirportSeqID", "Div4WheelsOn", "Div4TotalGTime", "Div4LongestGTime", "Div4WheelsOff", "Div4TailNum", "Div5Airport", "Div5AirportID", "Div5AirportSeqID", "Div5WheelsOn", "Div5TotalGTime", "Div5LongestGTime", "Div5WheelsOff", "Div5TailNum"
]

In [153]:
df = df.drop(columns=columns_to_drop, errors="ignore")

***
## Check for mixed data types

Check whether any column contains more than one Python data type, ignoring missing values. An empty result means each column has only one type in this file. It does not mean every type is ready for modeling—for example, `FlightDate` and scheduled times may still need to be converted.
***

In [154]:
mixed_type_columns = {}

for column in df.columns:
    type_counts = df[column].dropna().map(type).value_counts()
    if len(type_counts) > 1:
        mixed_type_columns[column] = type_counts

mixed_type_columns

{}

***
## Keep completed flights that were not diverted

Canceled flights do not have normal departure and arrival results. Diverted flights do not follow the planned route and schedule. Count both groups, then remove them so the remaining delay values describe comparable completed flights.

> A model trained on this data will predict delays only for flights that are not canceled or diverted. Predicting cancellations or diversions would require a separate dataset and target.
***

In [155]:
cancelled_or_diverted_counts = df[["Cancelled", "Diverted"]].sum().astype(int)
cancelled_or_diverted_counts

Cancelled    4652
Diverted      674
dtype: int64

In [156]:
df = df[(df["Cancelled"] != 1) & (df["Diverted"] != 1)].reset_index(drop=True)
df.shape

(241356, 47)

***
## Remove details that are not needed in the processed file

Remove the cancellation and diversion fields, along with detailed arrival, delay-cause, elapsed-time, and time-block fields. Display the columns that remain so the final file can be checked before it is saved.

> **Modeling warning:** this file still contains possible target columns and information recorded during or after a flight, including `DepTime`, departure-delay values, `TaxiOut`, `WheelsOff`, `ArrDel15`, and `ArrivalDelayGroups`. For a model that predicts before departure, choose one target and remove it, related versions of the same result, and anything that would not be known at prediction time from the model inputs.
***

In [157]:
columns_to_drop = [
    "WheelsOn",
    "TaxiIn",
    "ArrTime",
    "ArrDelay",
    "ArrDelayMinutes",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
    "FirstDepTime",
    "TotalAddGTime",
    "LongestAddGTime",
    "Cancelled",
    "CancellationCode",
    "Diverted",
    "ActualElapsedTime",
    "AirTime",
    "DepTimeBlk",
    "ArrTimeBlk"
]

In [158]:
df = df.drop(columns=columns_to_drop, errors="ignore")

In [159]:
df.columns

Index(['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate',
       'Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline',
       'Origin', 'OriginState', 'Dest', 'DestState', 'CRSDepTime', 'DepTime',
       'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups',
       'TaxiOut', 'WheelsOff', 'CRSArrTime', 'ArrDel15', 'ArrivalDelayGroups',
       'CRSElapsedTime', 'Distance', 'DistanceGroup'],
      dtype='object')

***
## Write the processed BTS file

Create the output folder if it does not exist, then save the processed dataframe without the pandas index. This is an intermediate file. Dates, times, missing values, categories, the target, and the final model inputs will be handled in later cleaning and modeling steps.
***

In [160]:
from pathlib import Path

Path(PROCESSED_DATA_FILE).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_DATA_FILE, index=False)